# 06 – EC3D Zero-Shot Retrieval (NO UNKNOWN)

**Versione paper-aligned**: 11 classi, senza Unknown.

In [1]:
USE_NO_UNKNOWN = True
N_CLASSES = 11
RESULTS_SUBDIR = 'NO_UNKNOWN'

import sys
from pathlib import Path
import pickle
import json
from datetime import datetime

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn.functional as F
from sklearn.metrics.pairwise import cosine_similarity

ROOT_DIR = Path('..').resolve()
sys.path.insert(0, str(ROOT_DIR))

from pose_encoder.twostream_stgcn_plus import TwoStreamSTGCNPlusEncoder
from text_encoder.distilbert_adapter import DistilBERTTextEncoder

# Add parent's parent for utils
sys.path.insert(0, str(ROOT_DIR.parent))
from utils import load_ec3d

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cuda


In [2]:
DATA_DIR = ROOT_DIR.parent / 'data' / 'EC3D'
ANNOTATIONS_DIR = ROOT_DIR.parent / 'data' / 'annotations_ec3d_no_unknown'
LOGS_DIR = ROOT_DIR.parent / 'logs'
RESULTS_DIR = ROOT_DIR.parent / 'results' / 'ec3d' / RESULTS_SUBDIR
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

ec3d_data = load_ec3d(DATA_DIR, no_unknown=USE_NO_UNKNOWN)
sequences = ec3d_data['sequences']
labels = np.array(ec3d_data['labels'])
test_indices = ec3d_data['test_indices']
ID_TO_NAME = ec3d_data['id_to_name']
SHORT_NAMES = ec3d_data['short_names']
NUM_CLASSES = ec3d_data['num_classes']

test_labels = labels[test_indices]
print(f'Test set: {len(test_indices)} sequences, {NUM_CLASSES} classes')

Test set: 88 sequences, 11 classes


In [3]:
# Load annotations
def load_generic_templates(path):
    templates = {}
    with open(path) as f:
        for i, line in enumerate(f):
            if line.strip(): templates[i] = line.strip()
    return templates

def load_flag3d_annotations(ann_dir, num_classes=11):
    anns = {i: [] for i in range(num_classes)}
    for cid in range(num_classes):
        for ann_idx in range(1, 7):
            fp = ann_dir / f'A{cid:03d}I{ann_idx:03d}.txt'
            if fp.exists():
                text = fp.read_text().strip()
                if text: anns[cid].append(text)
    return anns

generic_templates = load_generic_templates(ANNOTATIONS_DIR / 'generic_templates.txt')
flag3d_annotations = load_flag3d_annotations(ANNOTATIONS_DIR, NUM_CLASSES)
print(f'Loaded {len(generic_templates)} templates, {sum(len(v) for v in flag3d_annotations.values())} FLAG3D annotations')

Loaded 11 templates, 66 FLAG3D annotations


In [4]:
# Load encoders
with open(LOGS_DIR / 'best_epoch.txt') as f:
    BEST_EPOCH = int(f.read().strip())

pose_encoder = TwoStreamSTGCNPlusEncoder(input_dim=3, hidden_channels=[64,128,256,256], output_dim=128, num_nodes=25, dropout=0.1, fusion_dropout=0.3)
pose_encoder.load_state_dict(torch.load(LOGS_DIR / f'pose_encoder_epoch{BEST_EPOCH}.pt', map_location=device))
pose_encoder.to(device).eval()

text_encoder = DistilBERTTextEncoder(pretrained_model='distilbert-base-uncased', output_dim=128, freeze_bert=True, use_adapter=True, adapter_bottleneck=256)
text_encoder.load_state_dict(torch.load(LOGS_DIR / f'text_encoder_epoch{BEST_EPOCH}.pt', map_location=device))
text_encoder.to(device).eval()
print(f'Encoders loaded (epoch {BEST_EPOCH})')

🧊 DistilBERT congelato (100 parametri)
🔧 Adapter attivo (bottleneck=256)
📊 Text Encoder: 494,208 trainabili / 66,857,088 totali
Encoders loaded (epoch 30)


In [5]:
# Encoding functions
def encode_texts(text_encoder, texts, device, batch_size=32):
    embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        encoded = text_encoder.tokenizer(batch, padding=True, truncation=True, max_length=128, return_tensors='pt')
        with torch.no_grad():
            emb = text_encoder(encoded['input_ids'].to(device), encoded['attention_mask'].to(device))
            emb = F.normalize(emb, p=2, dim=1)
            embeddings.append(emb.cpu().numpy())
    return np.vstack(embeddings)

def encode_poses(pose_encoder, sequences, indices, device, max_len=150):
    embeddings = []
    for idx in tqdm(indices, desc='Encoding poses'):
        seq = np.transpose(sequences[idx], (0, 2, 1))
        T = seq.shape[0]
        if T < max_len:
            seq = np.concatenate([seq, np.zeros((max_len - T, 25, 3), dtype=np.float32)], axis=0)
        elif T > max_len:
            seq = seq[np.linspace(0, T-1, max_len).astype(int)]
        seq_t = torch.from_numpy(seq.astype(np.float32)).unsqueeze(0).to(device)
        with torch.no_grad():
            emb = pose_encoder(seq_t)
            embeddings.append(emb.cpu().numpy())
    return np.vstack(embeddings)

In [6]:
# Encode test poses
pose_embeddings = encode_poses(pose_encoder, sequences, test_indices, device)
print(f'Pose embeddings: {pose_embeddings.shape}')

Encoding poses:   0%|          | 0/88 [00:00<?, ?it/s]

Pose embeddings: (88, 128)


In [7]:
# Strategy 1: Generic templates
generic_texts = [generic_templates[i] for i in range(NUM_CLASSES)]
generic_embeddings = encode_texts(text_encoder, generic_texts, device)

sims_generic = cosine_similarity(pose_embeddings, generic_embeddings)
preds_generic = sims_generic.argmax(axis=1)
acc_generic = (preds_generic == test_labels).mean()
recall5_generic = np.mean([test_labels[i] in np.argsort(-sims_generic[i])[:5] for i in range(len(test_labels))])
print(f'GENERIC TEMPLATES: Recall@1={acc_generic:.4f}, Recall@5={recall5_generic:.4f}')

GENERIC TEMPLATES: Recall@1=0.0000, Recall@5=0.4205


In [8]:
# Strategy 2a: FLAG3D Mean
all_flag3d_texts = []
text_to_class = []
for cid, texts in flag3d_annotations.items():
    for t in texts:
        all_flag3d_texts.append(t)
        text_to_class.append(cid)
text_to_class = np.array(text_to_class)

all_flag3d_embeddings = encode_texts(text_encoder, all_flag3d_texts, device)

flag3d_mean_embeddings = np.zeros((NUM_CLASSES, 128))
for cid in range(NUM_CLASSES):
    mask = text_to_class == cid
    if mask.sum() > 0:
        mean_emb = all_flag3d_embeddings[mask].mean(axis=0)
        flag3d_mean_embeddings[cid] = mean_emb / np.linalg.norm(mean_emb)

sims_mean = cosine_similarity(pose_embeddings, flag3d_mean_embeddings)
preds_mean = sims_mean.argmax(axis=1)
acc_mean = (preds_mean == test_labels).mean()
recall5_mean = np.mean([test_labels[i] in np.argsort(-sims_mean[i])[:5] for i in range(len(test_labels))])
print(f'FLAG3D MEAN: Recall@1={acc_mean:.4f}, Recall@5={recall5_mean:.4f}')

FLAG3D MEAN: Recall@1=0.0000, Recall@5=0.5455


In [9]:
# Strategy 2b: FLAG3D Max
sims_all = cosine_similarity(pose_embeddings, all_flag3d_embeddings)
sims_max = np.zeros((len(test_labels), NUM_CLASSES))
for cid in range(NUM_CLASSES):
    mask = text_to_class == cid
    if mask.sum() > 0:
        sims_max[:, cid] = sims_all[:, mask].max(axis=1)

preds_max = sims_max.argmax(axis=1)
acc_max = (preds_max == test_labels).mean()
recall5_max = np.mean([test_labels[i] in np.argsort(-sims_max[i])[:5] for i in range(len(test_labels))])
print(f'FLAG3D MAX: Recall@1={acc_max:.4f}, Recall@5={recall5_max:.4f}')

FLAG3D MAX: Recall@1=0.0000, Recall@5=0.2841


In [10]:
# Save results
results = {
    'experiment': 'zero_shot_retrieval', 'version': 'NO_UNKNOWN', 'num_classes': NUM_CLASSES,
    'timestamp': datetime.now().isoformat(),
    'results': {
        'generic_templates': {'recall@1': float(acc_generic), 'recall@5': float(recall5_generic)},
        'flag3d_mean': {'recall@1': float(acc_mean), 'recall@5': float(recall5_mean)},
        'flag3d_max': {'recall@1': float(acc_max), 'recall@5': float(recall5_max)},
    }
}

with open(RESULTS_DIR / 'zero_shot_metrics.json', 'w') as f:
    json.dump(results, f, indent=2)

print(f'\nResults saved to {RESULTS_DIR}')


Results saved to /home/giov/Scrivania/Tesi/pose-text-feedback-thesis/results/ec3d/NO_UNKNOWN
